# Univariate ARIMA/SARIMA for hourly PM2.5 (London, 2024)

This notebook fits a classical autoregressive baseline on the training portion of an hourly univariate series and evaluates one-step-ahead-style multi-step forecasts on a held-out test window. The objective is to establish a transparent, interpretable benchmark prior to more flexible machine-learning approaches.

## Imports and project paths

Paths are resolved relative to the working directory so the notebook can be executed from the repository root or from the `notebooks` directory without manual edits.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller


def project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "london_2024.csv").exists():
        return cwd
    if (cwd.parent / "data" / "london_2024.csv").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not find data/london_2024.csv. "
        "Run with cwd = thesis_2026 or thesis_2026/notebooks."
    )


PROJECT_ROOT = project_root()
DATA_PATH = PROJECT_ROOT / "data" / "london_2024.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "arima_model.pkl"

In [ ]:
# Reproducibility: library versions and RNG seeds (copy output into thesis appendix if needed)
import random
import sys

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

try:
    import tensorflow as tf

    tf.random.set_seed(RNG_SEED)
except ImportError:
    pass

print("Python:", sys.version.split()[0])
for _name, _mod in (
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "sklearn"),
    ("statsmodels", "statsmodels"),
    ("pmdarima", "pmdarima"),
):
    try:
        _m = __import__(_mod)
        print(f"{_name}:", getattr(_m, "__version__", "?"))
    except ImportError:
        print(f"{_name}: (not installed)")
for _name, _mod in (("tensorflow", "tensorflow"), ("xgboost", "xgboost")):
    try:
        _m = __import__(_mod)
        print(f"{_name}:", getattr(_m, "__version__", "?"))
    except ImportError:
        print(f"{_name}: (not installed)")


## Data loading and panel preparation

For a univariate specification, identifiers that do not vary within this extract (`city`, `latitude`, `longitude`) carry no information and are removed. Observations are ordered in time, indexed at hourly frequency, and missing values are propagated forward along the time axis so that each hour inherits the last observed value—a simple monotone imputation that preserves the temporal index without introducing future information.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["time"])
df = df.drop(columns=["city", "latitude", "longitude"])
df = df.sort_values("time").set_index("time")
df = df.asfreq("h")
df = df.ffill()

## Target series and chronological train–test split

The analysis focuses on particulate matter (`pm2_5`) as the endogenous series. The sample is partitioned by time: the earliest segment trains the model and the latest segment provides strictly out-of-sample errors. Random shuffling would violate the forecasting setting and inflate apparent accuracy; the split used here is therefore index-preserving and temporal.

In [ ]:
y = df["pm2_5"].astype(float)
split = int(len(y) * 0.8)
y_train, y_test = y.iloc[:split], y.iloc[split:]
print(f"Train length: {len(y_train)}, test length: {len(y_test)}")
print(f"Train window: {y_train.index.min()} → {y_train.index.max()}")
print(f"Test window:  {y_test.index.min()} → {y_test.index.max()}")

## Stationarity: Augmented Dickey–Fuller test

The ADF procedure tests the null hypothesis that the series contains a unit root (non-stationary integrated behaviour). A small *p*-value constitutes evidence against that null, supporting covariance stationarity after appropriate differencing or in levels, depending on the data-generating process. The reported *p*-value guides the choice of differencing orders in Box–Jenkins modelling.

In [ ]:
adf_stat, adf_p, *_ = adfuller(y_train.dropna(), autolag="AIC")
print(f"ADF test statistic: {adf_stat:.6f}")
print(f"ADF p-value: {adf_p:.6e}")

## Seasonal ARIMA estimation

Hourly pollutant concentrations typically exhibit strong diurnal structure. A seasonal ARIMA component with period $m=24$ hours captures such cyclic mean reversion relative to a weekly seasonal alternative ($m=168$), which is considerably more demanding in estimation. Automatic order search (`auto_arima`) restricts the search space while remaining consistent with textbook SARIMA notation. Fitting can take several minutes on a full year of hourly observations.

In [ ]:
model = auto_arima(
    y_train,
    seasonal=True,
    m=24,
    suppress_warnings=True,
    stepwise=True,
    error_action="ignore",
)
print(f"Selected ARIMA order: {model.order}")
print(f"Selected seasonal order: {model.seasonal_order}")

## Out-of-sample accuracy

Root mean squared error (RMSE) penalises large errors more heavily than mean absolute error (MAE). Both metrics are reported on the held-out test horizon for comparability with alternative models evaluated under the same temporal protocol.

In [ ]:
y_pred = model.predict(n_periods=len(y_test))
y_pred = pd.Series(y_pred, index=y_test.index, name="pm2_5_pred")
mse = mean_squared_error(y_test, y_pred)
rmse = float(np.sqrt(mse))
mae = float(mean_absolute_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")

## Saving model metadata

A companion JSON file records the training timestamp, dataset version, and evaluation metrics alongside the model artifact. This enables version tracking without a full model registry.

In [ ]:
import json
from datetime import datetime, timezone

metadata = {
    "model": "arima",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "data_version": str(DATA_PATH.name),
    "train_rows": int(len(y_train)),
    "test_rows": int(len(y_test)),
    "arima_order": list(model.order),
    "seasonal_order": list(model.seasonal_order),
    "metrics": {"rmse": round(rmse, 4), "mae": round(mae, 4)},
    "notes": "Seasonal ARIMA fitted via pmdarima auto_arima (m=24).",
}
meta_path = MODEL_PATH.parent / "arima_metadata.json"
with meta_path.open("w") as fh:
    json.dump(metadata, fh, indent=2)
print(f"Saved metadata to {meta_path}")
print(json.dumps(metadata, indent=2))